## Project layout

Resolve the repo subdirectories so the rest of the notebook is
cwd-independent and figures land in the shared `figures/` folder.


In [ ]:
# Project layout: notebooks live in <root>/notebooks/, source in
# <root>/src/, persistent datasets in <root>/data/, saved figures in
# <root>/figures/. We resolve these once so the rest of the notebook is
# cwd-independent.
from pathlib import Path
import sys

REPO_ROOT   = Path.cwd().resolve().parent
SRC_DIR     = REPO_ROOT / 'src'
DATA_DIR    = REPO_ROOT / 'data'
FIGURES_DIR = REPO_ROOT / 'figures'
FIGURES_DIR.mkdir(exist_ok=True)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


# Geometry & seal sensitivity sweep

Tests robustness of the main-sweep conclusions to the geometric parameters
that are not well-constrained by the seismic / surface observations:

1. **Mud body width** (`mud_width`) — varied around the constrained value.
2. **Depth to overpressure source** (`depth_to_source`) — varied even though
   constrained from seismic, to bracket picking uncertainty.
3. **Lateral extent of overpressure** (`ovp_lateral_extent`) — width of the
   centrally-overpressured portion of the basal layer.
4. **Domain width** (`Lx`) — to check that the lateral boundaries are far
   enough away to not influence the centre.
5. **Intermediate seal** (`seal_thickness`, `seal_depth`, `seal_k`) — adds a
   thin low-k interbed within the otherwise-uniform sand column to test how
   a continuous secondary seal would change the response.

All sensitivity runs are performed at a single representative baseline
(taken from the centre of the admissible region in the main sweep).
Results are written to a separate HDF5 file so the main sweep is untouched.

In [ ]:
import time
import warnings

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from pockmark_model import PERM_FACTOR, run_simulation
from sweep_utils import (
    build_sample_cols, fmt_duration, load_done_keys,
    run_key_str, write_run_to_hdf5,
)

warnings.filterwarnings('ignore')

## Baseline parameter combination

Pick a single (`overpressure_mpa`, `k_mud`, `k_sand`, `ovp_thickness`) point
from the centre of the admissible region in the main sweep. Edit these to
match whichever combination you want to test for geometric robustness.

In [ ]:
BASELINE = dict(
    overpressure_mpa = 1.55,
    k_mud            = 1e-19 * PERM_FACTOR,
    k_sand           = 1e-11 * PERM_FACTOR,
    ovp_thickness    = 30.0,
    bulkrho_mud      = 1650.0,
    bulkrho_sand     = 2150.0,
    ss               = 1e-4,
    sy               = 0.15,
)

DEFAULT_GEOM = dict(
    Lx                 = 105_000.0,
    mud_width          = 35_000.0,
    depth_to_source    = 300.0,
    ovp_lateral_extent = None,
    seal_thickness     = 0.0,
    seal_depth         = 150.0,
    seal_k             = None,
)

HDF5_FILE = str(DATA_DIR / 'geometry_sensitivity_profiles.h5')

## Build sensitivity grid

Each axis is varied independently around the baseline geometry (one-at-a-time).
For the seal axis a small joint sweep over depth × thickness × k is included,
since these three interact.

In [ ]:
# One-axis-at-a-time perturbations
perturbations = {
    'mud_width':          [15_000.0, 25_000.0, 35_000.0, 45_000.0, 60_000.0],
    'depth_to_source':    [150.0, 225.0, 300.0, 400.0, 500.0],
    'ovp_lateral_extent': [10_000.0, 25_000.0, 50_000.0, 75_000.0, None],
    'Lx':                 [70_000.0, 105_000.0, 140_000.0, 175_000.0],
}

runs = []
for axis, values in perturbations.items():
    for v in values:
        runs.append((axis, {**DEFAULT_GEOM, axis: v}))

# Seal: joint depth × thickness × k
for sd in [75.0, 150.0, 225.0]:
    for st in [5.0, 10.0]:
        for sk_m2 in [1e-19, 1e-18, 1e-17]:
            runs.append(('seal', {
                **DEFAULT_GEOM,
                'seal_depth':     sd,
                'seal_thickness': st,
                'seal_k':         sk_m2 * PERM_FACTOR,
            }))

print(f'Sensitivity sweep: {len(runs)} runs.')

In [ ]:
def _key(axis, geom):
    return run_key_str(axis=axis, **geom)

def _run_one(axis, geom):
    try:
        res, _cm, _cs, grad_sp, x_c, in_mud = run_simulation(**BASELINE, **geom)
        return axis, geom, res, grad_sp, x_c, in_mud, None
    except Exception as e:
        return axis, geom, None, None, None, None, str(e)

done_keys = load_done_keys(HDF5_FILE)
runs_todo = [r for r in runs if _key(*r) not in done_keys]
print(f'{len(done_keys)} already done, {len(runs_todo)} remaining.')

In [ ]:
failed = {}
n_success = 0
start = time.time()

if runs_todo:
    with h5py.File(HDF5_FILE, 'a') as hf:
        if 'runs' not in hf:
            hf.attrs['description'] = 'Geometry / seal sensitivity sweep at fixed baseline'
            for k, v in BASELINE.items():
                hf.attrs[f'baseline_{k}'] = v
            hf.create_group('runs')
        for n, result in enumerate(
            Parallel(n_jobs=4, backend='loky', verbose=0, return_as='generator')(
                delayed(_run_one)(*r) for r in runs_todo
            ), start=1
        ):
            axis, geom, res, grad_sp, x_c, in_mud, err = result
            if err is not None:
                failed[(axis, _key(axis, geom))] = err
                continue
            sample_cols = build_sample_cols(x_c)
            params = {'axis': axis, **geom}
            write_run_to_hdf5(
                hf, _key(axis, geom), params,
                res['time'].values,
                np.asarray(grad_sp)[:, sample_cols],
                x_coords=x_c[sample_cols],
                in_mud_zone=in_mud[sample_cols],
            )
            n_success += 1
            elapsed = time.time() - start
            print(f'  [{n}/{len(runs_todo)}] {axis}  elapsed {fmt_duration(elapsed)}')
    print(f'Done. Successful {n_success}, failed {len(failed)}.')
else:
    print('All sensitivity runs already complete.')

## Sensitivity summary

For each run, compute first and last exceedance year for mud and sand.

In [ ]:
RHO_SW = 1024.0
crit_mud  = (BASELINE['bulkrho_mud']  - RHO_SW) / RHO_SW
crit_sand = (BASELINE['bulkrho_sand'] - RHO_SW) / RHO_SW

rows = []
with h5py.File(HDF5_FILE, 'r') as hf:
    for grp_name in hf['runs']:
        g = hf['runs'][grp_name]
        times  = g['time_yr'][:]
        grads  = g['gradients'][:].astype(np.float32)
        in_mud = g['in_mud_zone'][:]
        g_mud  = np.where(in_mud,  grads, 0.0).max(axis=1)
        g_sand = np.where(~in_mud, grads, 0.0).max(axis=1)
        em_t = times[g_mud  > crit_mud]
        es_t = times[g_sand > crit_sand]
        first_mud  = float(em_t[0])   if len(em_t)  else float('nan')
        last_mud   = float(em_t[-1])  if len(em_t)  else float('nan')
        first_sand = float(es_t[0])   if len(es_t)  else float('nan')
        last_sand  = float(es_t[-1])  if len(es_t)  else float('nan')
        rows.append({
            'group': grp_name,
            'peak_grad_mud':        float(g_mud.max()),
            'peak_grad_sand':       float(g_sand.max()),
            'first_exceed_mud_yr':  first_mud,
            'last_exceed_mud_yr':   last_mud,
            'first_exceed_sand_yr': first_sand,
            'last_exceed_sand_yr':  last_sand,
            **{k: g.attrs[k] for k in g.attrs},
        })

df_sens = pd.DataFrame(rows)
df_sens.to_csv(DATA_DIR / 'geometry_sensitivity_summary.csv', index=False)
df_sens.groupby('axis')[['first_exceed_mud_yr', 'last_exceed_mud_yr',
                          'first_exceed_sand_yr', 'last_exceed_sand_yr']].mean()


## Plot: first and last exceedance year vs each one-at-a-time axis

For each sensitivity axis, shows how first and last exceedance year for mud
and sand change as the parameter varies around the baseline.
Solid lines = first exceedance; dashed lines = last exceedance.

In [ ]:
axes_to_plot = [
    ('mud_width',          'Mud-body width [m]'),
    ('depth_to_source',    'Depth to source [m]'),
    ('ovp_lateral_extent', 'OVP lateral extent [m]'),
    ('Lx',                 'Domain width [m]'),
]

MUD_COLOR  = '#1f77b4'
SAND_COLOR = '#ff7f0e'
OVP_FULL_DOMAIN_M = DEFAULT_GEOM['Lx']

fig, axs = plt.subplots(2, 2, figsize=(11, 7), constrained_layout=True)
for ax, (axis, label) in zip(axs.flat, axes_to_plot):
    sub = df_sens[df_sens['axis'] == axis].copy()
    if sub.empty:
        ax.text(0.5, 0.5, 'no data', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(label); continue

    if axis == 'ovp_lateral_extent':
        # -1 stored as sentinel for None (full domain); remap BEFORE sorting
        sub['_x'] = sub[axis].astype(float).where(sub[axis] >= 0, OVP_FULL_DOMAIN_M)
        sub = sub.sort_values('_x')
        x_plot = sub['_x'].values
        finite_vals = sorted(sub.loc[sub[axis] >= 0, axis].unique())
        tick_vals   = list(finite_vals) + [OVP_FULL_DOMAIN_M]
        tick_labels = [f'{int(v):,}' for v in finite_vals] + [f'{int(OVP_FULL_DOMAIN_M):,}\n(full domain)']
    else:
        sub = sub.sort_values(axis)
        x_plot = sub[axis].values
        tick_vals, tick_labels = None, None

    ax.plot(x_plot, sub['first_exceed_mud_yr'],  '-o', color=MUD_COLOR,
            label='Mud first exceedance',  markersize=6)
    ax.plot(x_plot, sub['last_exceed_mud_yr'],   '--o', color=MUD_COLOR,
            label='Mud last exceedance',   markersize=6, alpha=0.6)
    ax.plot(x_plot, sub['first_exceed_sand_yr'], '-s', color=SAND_COLOR,
            label='Sand first exceedance', markersize=6)
    ax.plot(x_plot, sub['last_exceed_sand_yr'],  '--s', color=SAND_COLOR,
            label='Sand last exceedance',  markersize=6, alpha=0.6)

    if tick_vals is not None:
        ax.set_xticks(tick_vals)
        ax.set_xticklabels(tick_labels, fontsize=8)

    ax.set_xlabel(label)
    ax.set_ylabel('Exceedance year [yr]')
    ax.legend(fontsize=7, ncol=2)
    ax.set_title(label)

fig.suptitle('Geometry sensitivity — first and last exceedance year (mud and sand)')
plt.savefig(FIGURES_DIR / 'fig_geom_sens_exceedance_axes.png', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Seal sensitivity: first and last exceedance year for mud and sand
# Grid: rows = seal depth, cols = mud | sand
# Lines: solid = first exceedance, dashed = last exceedance
# Hue:   seal thickness (5 m vs 10 m)
seal_df = df_sens[df_sens['axis'] == 'seal'].copy()
if not seal_df.empty:
    seal_df['log_k'] = np.log10(seal_df['seal_k'] / PERM_FACTOR)
    depths      = sorted(seal_df['seal_depth'].unique())
    thicknesses = sorted(seal_df['seal_thickness'].unique())
    THICK_COLORS = {t: c for t, c in zip(thicknesses, ['#2196F3', '#E91E63'])}

    MUD_COLOR  = '#1f77b4'
    SAND_COLOR = '#ff7f0e'

    nrows = len(depths)
    fig, axs = plt.subplots(nrows, 2, figsize=(10, 3.2 * nrows),
                             constrained_layout=True, sharey='row', sharex=True)

    for row, depth in enumerate(depths):
        for col, (first_col, last_col, material, color) in enumerate([
            ('first_exceed_mud_yr',  'last_exceed_mud_yr',  'Mud',  MUD_COLOR),
            ('first_exceed_sand_yr', 'last_exceed_sand_yr', 'Sand', SAND_COLOR),
        ]):
            ax = axs[row, col]
            sub_d = seal_df[seal_df['seal_depth'] == depth]
            for thickness in thicknesses:
                sub = sub_d[sub_d['seal_thickness'] == thickness].sort_values('log_k')
                if sub.empty:
                    continue
                c = THICK_COLORS[thickness]
                ax.plot(sub['log_k'], sub[first_col], '-o', color=c,
                        markersize=5, label=f'{int(thickness)} m seal — first')
                ax.plot(sub['log_k'], sub[last_col],  '--o', color=c,
                        markersize=5, alpha=0.55, label=f'{int(thickness)} m seal — last')
            ax.set_title(f'{material}  |  seal depth {int(depth)} m', fontsize=9)
            ax.set_ylabel('Exceedance year [yr]', fontsize=8)
            if row == nrows - 1:
                ax.set_xlabel('log₁₀(seal k [m²])', fontsize=8)
            if row == 0 and col == 1:
                ax.legend(fontsize=7, loc='upper left',
                          title='Seal thickness / exceedance', title_fontsize=7)

    fig.suptitle('Effect of intermediate seal layer on exceedance year\n'
                 '(solid = first exceedance, dashed = last exceedance)', fontsize=11)
    plt.savefig(FIGURES_DIR / 'fig_geom_sens_seal.png', dpi=200, bbox_inches='tight')
    plt.show()
else:
    print('No seal runs in this sensitivity dataset.')
